# Experiment: Unsupervised SimCSE Training (MPNet + Hard Negatives)
Objective:
- Base model: `sentence-transformers/all-mpnet-base-v2`
- Stage 1: unsupervised SimCSE on `data_new/SNow_incident_ticket_data.csv`
- Stage 2: hard-negative refinement using false positives from eval pairs
- Compare baseline vs stage1 vs stage2 and write artifacts


In [7]:
from __future__ import annotations
import json, random, re
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd, torch
from scipy.stats import spearmanr, pearsonr
from sentence_transformers import SentenceTransformer, InputExample, losses
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score, ndcg_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from torch.utils.data import DataLoader

def ts():
    return datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

def root():
    cwd = Path.cwd()
    if (cwd / "data_new").exists():
        return cwd
    if (cwd / "nexustism" / "data_new").exists():
        return cwd / "nexustism"
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = root()
RUN_TS = ts()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIG = {
    "seed": 42,
    "paths": {
        "train_csv": "data_new/SNow_incident_ticket_data.csv",
        "eval_pairs_json": "data_new/fixed_test_pairs.json",
        "run_output_root": f"models/unsupervised_simcse/runs/{RUN_TS}",
        "simcse_parent": "models/unsupervised_simcse",
    },
    "model": {"base_model_id": "sentence-transformers/all-mpnet-base-v2"},
    "eval": {"calibration_fraction": 0.3, "batch_size": 32, "bootstrap_iters": 200, "k_values": [5, 10]},
    "train": {
        "stage1": {"run": True, "batch_size": 64, "epochs": 1, "lr": 2e-5, "warmup_ratio": 0.1, "max_seq_length": 256, "max_train_sentences": 30000},
        "stage2_hardneg": {"run": True, "batch_size": 32, "epochs": 1, "lr": 1e-5, "warmup_ratio": 0.1, "max_seq_length": 256, "max_hard_negatives": 300, "max_refinement_pairs": 2000},
    },
    "privacy": {"max_words": 180, "min_words": 4},
}
random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CONFIG["seed"])
print("PROJECT_ROOT", PROJECT_ROOT); print("DEVICE", DEVICE); print("RUN_TS", RUN_TS)


PROJECT_ROOT c:\Users\donov\Documents\itsm project\Final_nexustism\nexustism
DEVICE cuda
RUN_TS 20260212_091123


In [8]:
def clean_text(t: str, max_words: int = 180) -> str:
    t = str(t)
    t = re.sub(r"https?://\\S+|www\\.\\S+", " ", t, flags=re.I)
    t = re.sub(r"\\b(?:INC|RITM|REQ|CHG|PRB|TASK)\\d+\\b", " ", t, flags=re.I)
    t = re.sub(r"[\\r\\n\\t]+", " ", t)
    t = re.sub(r"\\s+", " ", t).strip()
    if not t: return ""
    w = t.split(" ")
    return " ".join(w[:max_words])

def first(paths):
    for p in paths:
        if p.exists(): return p
    raise FileNotFoundError(paths)

def load_train():
    p = first([PROJECT_ROOT / CONFIG["paths"]["train_csv"], PROJECT_ROOT / "data" / "SNow_incident_ticket_data.csv"])
    df = pd.read_csv(p)
    col = next((c for c in ["Description", "description", "cleaned_text", "text"] if c in df.columns), None)
    if not col: raise KeyError(f"No text column in {p}")
    txt = [clean_text(x, CONFIG["privacy"]["max_words"]) for x in df[col].fillna("").astype(str).tolist()]
    txt = [x for x in txt if x and len(x.split()) >= int(CONFIG["privacy"]["min_words"])]
    uniq, seen = [], set()
    for x in txt:
        if x in seen: continue
        uniq.append(x); seen.add(x)
    profile = {"path": str(p), "text_col": col, "raw_rows": int(len(df)), "normalized_non_empty": int(len(txt)), "unique_non_empty": int(len(uniq))}
    return uniq, profile

def load_pairs():
    p = first([PROJECT_ROOT / CONFIG["paths"]["eval_pairs_json"], PROJECT_ROOT / "data" / "fixed_test_pairs.json"])
    d = json.loads(p.read_text(encoding="utf-8"))
    t1 = [str(x) for x in d.get("texts1", [])]
    t2 = [str(x) for x in d.get("texts2", [])]
    y = np.asarray([int(x) for x in d.get("labels", [])], dtype=np.int64)
    if not (len(t1) == len(t2) == len(y)): raise ValueError("pair length mismatch")
    if len(y) == 0: raise ValueError("empty pairs")
    return {"path": str(p), "texts1": t1, "texts2": t2, "labels": y}

TRAIN_TEXTS, TRAIN_PROFILE = load_train()
PAIRS = load_pairs()
print(TRAIN_PROFILE); print("pairs", len(PAIRS["labels"]), "pos_ratio", float(np.mean(PAIRS["labels"])))


{'path': 'c:\\Users\\donov\\Documents\\itsm project\\Final_nexustism\\nexustism\\data_new\\SNow_incident_ticket_data.csv', 'text_col': 'Description', 'raw_rows': 10633, 'normalized_non_empty': 10498, 'unique_non_empty': 10278}
pairs 1000 pos_ratio 0.5


In [9]:
def split_pairs(payload, frac=0.3, seed=42):
    y = payload["labels"]; idx = np.arange(len(y)); strat = y if np.unique(y).size > 1 else None
    try:
        c, e = train_test_split(idx, train_size=float(frac), random_state=int(seed), stratify=strat)
    except Exception:
        k = max(1, int(len(idx) * float(frac))); c, e = idx[:k], idx[k:] if k < len(idx) else idx[-1:]
    def pack(ii):
        ii = [int(i) for i in ii]
        return {"texts1": [payload["texts1"][i] for i in ii], "texts2": [payload["texts2"][i] for i in ii], "labels": np.asarray([int(payload["labels"][i]) for i in ii], dtype=np.int64), "idx": ii}
    cal, ev = pack(c), pack(e)
    if len(cal["labels"]) == 0 or len(ev["labels"]) == 0: raise RuntimeError("empty split")
    return cal, ev

def cosine_scores(model, a, b, bs=32):
    e1 = model.encode(a, batch_size=int(bs), convert_to_numpy=True, show_progress_bar=False)
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    e2 = model.encode(b, batch_size=int(bs), convert_to_numpy=True, show_progress_bar=False)
    e1 = normalize(np.asarray(e1, dtype=np.float32), norm="l2")
    e2 = normalize(np.asarray(e2, dtype=np.float32), norm="l2")
    return np.sum(e1 * e2, axis=1).astype(np.float64)

def boot(metric, y, s, iters=200, seed=42):
    rng = np.random.default_rng(seed); n = len(y); out = []
    for _ in range(int(iters)):
        ii = rng.integers(0, n, size=n)
        try: out.append(float(metric(y[ii], s[ii])))
        except Exception: pass
    if not out: return [float("nan"), float("nan")]
    z = np.asarray(out, dtype=np.float64)
    return [float(np.percentile(z, 2.5)), float(np.percentile(z, 97.5))]

def best_thr(y, s):
    bt, bf = 0.0, -1.0
    for t in np.linspace(-1.0, 1.0, 401):
        f = f1_score(y, (s >= t).astype(int))
        if f > bf: bt, bf = float(t), float(f)
    return bt, bf

def metrics(y, s, thr, iters=200, seed=42):
    p = (s >= thr).astype(int)
    return {
        "spearman": float(spearmanr(s, y)[0]),
        "pearson": float(pearsonr(s, y)[0]),
        "roc_auc": float(roc_auc_score(y, s)),
        "pr_auc": float(average_precision_score(y, s)),
        "f1": float(f1_score(y, p)),
        "accuracy": float(accuracy_score(y, p)),
        "threshold": float(thr),
        "spearman_ci": boot(lambda yy, ss: spearmanr(ss, yy)[0], y, s, iters, seed),
        "roc_auc_ci": boot(roc_auc_score, y, s, iters, seed),
    }

def eval_model(model, cal, ev):
    bs = int(CONFIG["eval"]["batch_size"])
    cs = cosine_scores(model, cal["texts1"], cal["texts2"], bs)
    es = cosine_scores(model, ev["texts1"], ev["texts2"], bs)
    thr, bf = best_thr(cal["labels"], cs)
    cm = metrics(cal["labels"], cs, thr, CONFIG["eval"]["bootstrap_iters"], CONFIG["seed"]); cm["best_calibration_f1"] = bf
    em = metrics(ev["labels"], es, thr, CONFIG["eval"]["bootstrap_iters"], CONFIG["seed"])
    return {"calibration": cm, "evaluation": em}, cs, es

CAL, EV = split_pairs(PAIRS, CONFIG["eval"]["calibration_fraction"], CONFIG["seed"])
print("cal", len(CAL["labels"]), "ev", len(EV["labels"]))


cal 300 ev 700


## Stage 1 + Stage 2 Training
- Stage 1 uses `MultipleNegativesRankingLoss`
- Stage 2 mines hard negatives and refines with `CosineSimilarityLoss`


In [10]:
def load_model(path, local_only=False):
    return SentenceTransformer(path, device=str(DEVICE), local_files_only=bool(local_only), trust_remote_code=False)

def train_stage1():
    cfg = CONFIG["train"]["stage1"]
    out = PROJECT_ROOT / CONFIG["paths"]["simcse_parent"] / f"simcse_{RUN_TS}"
    out.parent.mkdir(parents=True, exist_ok=True)
    m = load_model(CONFIG["model"]["base_model_id"], local_only=False)
    m.max_seq_length = int(cfg["max_seq_length"])
    txt = TRAIN_TEXTS[: int(cfg["max_train_sentences"])]
    if len(txt) < 2:
        raise RuntimeError("not enough training text")
    ex = [InputExample(texts=[t, t]) for t in txt]
    dl = DataLoader(ex, batch_size=int(cfg["batch_size"]), shuffle=True, drop_last=(len(ex) >= int(cfg["batch_size"])))
    loss = losses.MultipleNegativesRankingLoss(m)
    steps = max(1, len(dl) * int(cfg["epochs"]))
    m.fit(
        train_objectives=[(dl, loss)],
        epochs=int(cfg["epochs"]),
        warmup_steps=int(steps * float(cfg["warmup_ratio"])),
        optimizer_params={"lr": float(cfg["lr"])} ,
        output_path=str(out),
        show_progress_bar=False,
        use_amp=(DEVICE.type == "cuda"),
    )
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

def mine_hardneg(source_payload, source_scores, threshold):
    y = source_payload["labels"]
    idx = np.where((y == 0) & (source_scores >= float(threshold)))[0]
    s = sorted(idx.tolist(), key=lambda i: float(source_scores[i]), reverse=True)[: int(CONFIG["train"]["stage2_hardneg"]["max_hard_negatives"])]
    return {
        "source_split": "calibration",
        "threshold": float(threshold),
        "total_false_positives": int(len(idx)),
        "selected_hard_negatives": int(len(s)),
        "selected_indices": [int(i) for i in s],
    }

def refine_pairs(source_payload, hidx):
    y = source_payload["labels"]
    t1 = source_payload["texts1"]
    t2 = source_payload["texts2"]
    pos = np.where(y == 1)[0].tolist()
    if not pos:
        raise RuntimeError("no positives for stage2")
    qp = defaultdict(list)
    for i in pos:
        qp[t1[i]].append(i)
    rng = random.Random(CONFIG["seed"])
    out = []
    limit = int(CONFIG["train"]["stage2_hardneg"]["max_refinement_pairs"])
    for i in hidx:
        q = t1[i]
        pidx = rng.choice(qp[q]) if qp.get(q) else rng.choice(pos)
        out.append({"text1": q, "text2": t2[pidx], "label": 1.0, "source": "positive_for_hard_query"})
        out.append({"text1": q, "text2": t2[i], "label": 0.0, "source": "hard_negative"})
        if len(out) >= limit:
            break
    while len(out) < limit:
        i = rng.choice(pos)
        out.append({"text1": t1[i], "text2": t2[i], "label": 1.0, "source": "random_positive"})
    return out[:limit]

def train_stage2(stage1_dir, pairs):
    cfg = CONFIG["train"]["stage2_hardneg"]
    out = PROJECT_ROOT / CONFIG["paths"]["simcse_parent"] / f"simcse_{RUN_TS}_hardneg"
    m = load_model(str(stage1_dir), local_only=True)
    m.max_seq_length = int(cfg["max_seq_length"])
    ex = [InputExample(texts=[r["text1"], r["text2"]], label=float(r["label"])) for r in pairs]
    dl = DataLoader(ex, batch_size=int(cfg["batch_size"]), shuffle=True, drop_last=(len(ex) >= int(cfg["batch_size"])))
    loss = losses.CosineSimilarityLoss(m)
    steps = max(1, len(dl) * int(cfg["epochs"]))
    m.fit(
        train_objectives=[(dl, loss)],
        epochs=int(cfg["epochs"]),
        warmup_steps=int(steps * float(cfg["warmup_ratio"])),
        optimizer_params={"lr": float(cfg["lr"])} ,
        output_path=str(out),
        show_progress_bar=False,
        use_amp=(DEVICE.type == "cuda"),
    )
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

base = load_model(CONFIG["model"]["base_model_id"], local_only=False)
base_m, base_cal_scores, base_ev_scores = eval_model(base, CAL, EV)
del base
if torch.cuda.is_available():
    torch.cuda.empty_cache()

STAGE1_DIR = train_stage1() if CONFIG["train"]["stage1"]["run"] else None
s1 = load_model(str(STAGE1_DIR), local_only=True)
s1_m, s1_cal_scores, s1_ev_scores = eval_model(s1, CAL, EV)
del s1
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# No leakage: mine and build refinement pairs from CAL only.
HARDNEG_STATS = mine_hardneg(CAL, s1_cal_scores, s1_m["calibration"]["threshold"])
REFINE = refine_pairs(CAL, HARDNEG_STATS["selected_indices"])

STAGE2_DIR = train_stage2(STAGE1_DIR, REFINE) if CONFIG["train"]["stage2_hardneg"]["run"] else None
s2 = load_model(str(STAGE2_DIR), local_only=True)
s2_m, s2_cal_scores, s2_ev_scores = eval_model(s2, CAL, EV)
del s2
if torch.cuda.is_available():
    torch.cuda.empty_cache()

HARDNEG_STATS["refinement_pair_count"] = int(len(REFINE))
HARDNEG_STATS["refinement_label_distribution"] = {
    str(k): int(v) for k, v in pd.Series([r["label"] for r in REFINE]).value_counts().to_dict().items()
}
print("stage1", STAGE1_DIR)
print("stage2", STAGE2_DIR)
print("hardneg", HARDNEG_STATS["selected_hard_negatives"], "from", HARDNEG_STATS["source_split"])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'train_runtime': '49.13', 'train_samples_per_second': '208.4', 'train_steps_per_second': '3.257', 'train_loss': '0.01462', 'epoch': '1'}


... (4 outputs left out)

{'train_runtime': '10.36', 'train_samples_per_second': '191.5', 'train_steps_per_second': '5.985', 'train_loss': '0.06937', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

stage1 c:\Users\donov\Documents\itsm project\Final_nexustism\nexustism\models\unsupervised_simcse\simcse_20260212_091123
stage2 c:\Users\donov\Documents\itsm project\Final_nexustism\nexustism\models\unsupervised_simcse\simcse_20260212_091123_hardneg
hardneg 59 from calibration


In [11]:
def ranking_metrics(payload, scores, k_values=(5, 10)):
    grp = defaultdict(list)
    for i, (q, c, y, s) in enumerate(zip(payload["texts1"], payload["texts2"], payload["labels"], scores)):
        grp[str(q)].append((int(y), float(s), i, str(c)))
    n_q = len(grp)
    with_pos = 0
    sizes = []
    single = 0
    mrr = []
    nd = {int(k): [] for k in k_values}
    rc = {int(k): [] for k in k_values}
    for rows in grp.values():
        rows = sorted(rows, key=lambda x: x[1], reverse=True)
        sizes.append(len(rows))
        single += int(len(rows) <= 1)
        y = np.asarray([r[0] for r in rows], dtype=np.int64)
        s = np.asarray([r[1] for r in rows], dtype=np.float64)
        if y.sum() <= 0:
            continue
        with_pos += 1
        first = int(np.where(y == 1)[0][0]) + 1
        mrr.append(1.0 / first)
        for k in k_values:
            k = int(k)
            rc[k].append(float(y[:k].sum() > 0))
            try:
                nd[k].append(float(ndcg_score([y], [s], k=k)))
            except Exception:
                nd[k].append(float("nan"))
    out = {
        "n_queries": int(n_q),
        "queries_with_positive": int(with_pos),
        "mean_group_size": float(np.mean(sizes)) if sizes else 0.0,
        "max_group_size": int(max(sizes)) if sizes else 0,
        "single_candidate_query_ratio": float(single / n_q) if n_q else 0.0,
        "mrr": float(np.nanmean(mrr)) if mrr else 0.0,
    }
    for k in k_values:
        k = int(k)
        out[f"ndcg@{k}"] = float(np.nanmean(nd[k])) if nd[k] else 0.0
        out[f"recall@{k}"] = float(np.nanmean(rc[k])) if rc[k] else 0.0
    return out

def paired_sig(y, a, b, metric="spearman", iters=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y)
    d = []
    for _ in range(int(iters)):
        ii = rng.integers(0, n, size=n)
        yy, aa, bb = y[ii], a[ii], b[ii]
        try:
            va = float(spearmanr(aa, yy)[0]) if metric == "spearman" else float(roc_auc_score(yy, aa))
            vb = float(spearmanr(bb, yy)[0]) if metric == "spearman" else float(roc_auc_score(yy, bb))
            d.append(va - vb)
        except Exception:
            pass
    z = np.asarray(d, dtype=np.float64)
    if z.size == 0:
        return {"delta": float("nan"), "p_value": float("nan"), "ci": [float("nan"), float("nan")], "significant": False}
    p = float(2.0 * min(np.mean(z >= 0), np.mean(z <= 0)))
    return {
        "delta": float(np.mean(z)),
        "p_value": p,
        "ci": [float(np.percentile(z, 2.5)), float(np.percentile(z, 97.5))],
        "significant": bool(p < 0.05),
    }

MODEL_METRICS = {
    "baseline_mpnet": base_m,
    "stage1_simcse": s1_m,
    "stage2_hardneg": s2_m,
}

# Held-out evaluation only.
MODEL_EVAL_SCORES = {
    "baseline_mpnet": base_ev_scores,
    "stage1_simcse": s1_ev_scores,
    "stage2_hardneg": s2_ev_scores,
}

comp = []
for n, p in MODEL_METRICS.items():
    ev = p["evaluation"]
    comp.append({
        "model_name": n,
        "spearman": ev["spearman"],
        "roc_auc": ev["roc_auc"],
        "pr_auc": ev["pr_auc"],
        "f1": ev["f1"],
        "accuracy": ev["accuracy"],
        "threshold": ev["threshold"],
    })

comparison_df = pd.DataFrame(comp).sort_values(["spearman", "roc_auc", "f1"], ascending=False).reset_index(drop=True)
RANK = {n: ranking_metrics(EV, s, CONFIG["eval"]["k_values"]) for n, s in MODEL_EVAL_SCORES.items()}
ranking_df = pd.DataFrame([{"model_name": n, **m} for n, m in RANK.items()]).sort_values("mrr", ascending=False)

top = comparison_df["model_name"].tolist()
SIGNIFICANCE = {
    "top_model": top[0],
    "runner_up": top[1] if len(top) > 1 else None,
    "evaluation_split_only": True,
    "spearman": paired_sig(EV["labels"], MODEL_EVAL_SCORES[top[0]], MODEL_EVAL_SCORES[top[1]], "spearman", 1000, CONFIG["seed"]) if len(top) > 1 else {},
    "roc_auc": paired_sig(EV["labels"], MODEL_EVAL_SCORES[top[0]], MODEL_EVAL_SCORES[top[1]], "roc_auc", 1000, CONFIG["seed"]) if len(top) > 1 else {},
}
comparison_df


,model_name,spearman,roc_auc,pr_auc,f1,accuracy,threshold
0,baseline_mpnet,0.483724,0.779273,0.822956,0.721585,0.618571,0.305
1,stage2_hardneg,0.384507,0.721992,0.776221,0.679245,0.635714,0.980
2,stage1_simcse,0.356115,0.705600,0.744249,0.640957,0.614286,0.275


## Decision
Decision rule:
1. Highest evaluation Spearman
2. Tie-break with ROC-AUC
3. Add paired-bootstrap significance note


In [12]:
OUT = PROJECT_ROOT / CONFIG["paths"]["run_output_root"]
OUT.mkdir(parents=True, exist_ok=True)

BEST_MODEL_NAME = comparison_df.iloc[0]["model_name"]
BEST_MODEL_PATH = {
    "baseline_mpnet": CONFIG["model"]["base_model_id"],
    "stage1_simcse": str(STAGE1_DIR),
    "stage2_hardneg": str(STAGE2_DIR),
}[BEST_MODEL_NAME]

assert (STAGE1_DIR / "config.json").exists() or (STAGE1_DIR / "model.safetensors").exists(), "Stage1 checkpoint missing"
assert (STAGE2_DIR / "config.json").exists() or (STAGE2_DIR / "model.safetensors").exists(), "Stage2 checkpoint missing"
assert str(STAGE1_DIR) != str(STAGE2_DIR), "Stage checkpoints must differ"
assert HARDNEG_STATS.get("source_split") == "calibration", "Hard negatives must come from calibration split"

for n, p in MODEL_METRICS.items():
    assert np.isfinite(p["evaluation"]["spearman"]), f"bad spearman {n}"
    assert np.isfinite(p["evaluation"]["roc_auc"]), f"bad roc_auc {n}"
for n, p in RANK.items():
    assert int(p["n_queries"]) > 0, f"no queries {n}"

metrics_payload = {
    "run_timestamp": RUN_TS,
    "project_root": str(PROJECT_ROOT),
    "device": str(DEVICE),
    "config": CONFIG,
    "training_profile": TRAIN_PROFILE,
    "checkpoints": {"stage1": str(STAGE1_DIR), "stage2": str(STAGE2_DIR)},
    "model_metrics": MODEL_METRICS,
    "comparison_table": comparison_df.to_dict(orient="records"),
    "significance": SIGNIFICANCE,
}
ranking_payload = {"ranking_metrics": RANK, "ranking_table": ranking_df.to_dict(orient="records")}
(OUT / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
(OUT / "ranking_metrics.json").write_text(json.dumps(ranking_payload, indent=2), encoding="utf-8")
(OUT / "hard_negative_stats.json").write_text(json.dumps(HARDNEG_STATS, indent=2), encoding="utf-8")

wr = comparison_df.iloc[0].to_dict()
summary = [
    "# Unsupervised SimCSE Training Summary",
    "",
    f"- Run timestamp: {RUN_TS}",
    f"- Device: {DEVICE}",
    f"- Base model: {CONFIG['model']['base_model_id']}",
    f"- Stage 1 checkpoint: {STAGE1_DIR}",
    f"- Stage 2 checkpoint: {STAGE2_DIR}",
    "",
    "## Winner",
    "",
    f"- Winner model: {wr['model_name']}",
    f"- Spearman: {wr['spearman']:.6f}",
    f"- ROC-AUC: {wr['roc_auc']:.6f}",
    "",
    "## Significance",
    "",
    f"- Comparison: {SIGNIFICANCE.get('top_model')} vs {SIGNIFICANCE.get('runner_up')}",
    f"- Spearman p-value: {SIGNIFICANCE.get('spearman', {}).get('p_value')}",
    f"- ROC-AUC p-value: {SIGNIFICANCE.get('roc_auc', {}).get('p_value')}",
]
(OUT / "summary.md").write_text("\n".join(summary) + "\n", encoding="utf-8")

print("BEST_MODEL_NAME", BEST_MODEL_NAME)
print("BEST_MODEL_PATH", BEST_MODEL_PATH)
print("Artifacts:", OUT)


BEST_MODEL_NAME baseline_mpnet
BEST_MODEL_PATH sentence-transformers/all-mpnet-base-v2
Artifacts: c:\Users\donov\Documents\itsm project\Final_nexustism\nexustism\models\unsupervised_simcse\runs\20260212_091123


## How To Deploy
```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(BEST_MODEL_PATH)
embeddings = model.encode(
    ["sample incident text 1", "sample incident text 2"],
    normalize_embeddings=True,
)
```
Persist for traceability:
- `metrics.json`
- `ranking_metrics.json`
- `hard_negative_stats.json`
- `summary.md`
